# Karta pracy: prosta klasyfikacja w `scikit-learn`

## Temat
Przewidywanie, czy student zaliczy przedmiot, na podstawie danych o aktywności i wynikach cząstkowych.

## Plik danych
Użyj pliku:

```text
student_pass_classification_dirty.csv
```

## Cel ćwiczenia
Twoim zadaniem jest przejście przez pełny proces budowy prostego modelu klasyfikacyjnego:

```text
CSV → wczytanie danych → sprawdzenie jakości danych → czyszczenie → preprocessing → X/y → train/test → model → predykcja → ocena
```

Zmienna docelowa: `passed`  
Możliwe klasy: `yes`, `no`

## 1. Import bibliotek

Zaimportuj biblioteki potrzebne do pracy z danymi, wizualizacji oraz budowy modelu klasyfikacyjnego.

Potrzebne będą między innymi: `pandas`, `numpy`, `matplotlib.pyplot`, `train_test_split`, `ColumnTransformer`, `OneHotEncoder`, `StandardScaler`, `Pipeline`, `LogisticRegression` oraz metryki oceny klasyfikacji.

In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pandas import read_csv
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

## 2. Wczytanie danych

Wczytaj plik CSV do zmiennej `df`.

Następnie wyświetl:
- pierwsze 5 wierszy,
- liczbę wierszy i kolumn,
- nazwy kolumn.

In [16]:
df = pd.read_csv("student_pass_classification_dirty.csv")
print(df.head())
print(df.shape)
print(df.columns)

  student_id  age  gender     city      program study_mode learning_platform  \
0    STU0001   23   other  Wroclaw  Informatics  part-time  Google Classroom   
1    STU0002   24  female   Krakow   Management  part-time             Teams   
2    STU0003   27  female   Gdansk   Management  part-time            Moodle   
3    STU0004   23   other     Lodz      Finance  full-time            Moodle   
4    STU0005   26    male   Warsaw   Management  part-time             Teams   

   attendance_percent  weekly_study_hours  assignment_score  quiz_score  \
0                48.0                16.7             100.0        91.0   
1                43.0                 4.6              48.9         9.3   
2                94.0                 6.7              72.8        51.8   
3                57.0                 4.9              74.0        58.4   
4                50.0                11.0              71.9        66.1   

   previous_grade  forum_posts project_submitted passed  
0         

## 3. Rozpoznanie struktury danych

Sprawdź strukturę danych.

Wykonaj:
- `info()`,
- `describe()` dla kolumn liczbowych,
- `describe(include="str")` dla kolumn tekstowych.

Następnie odpowiedz krótko na pytania:
1. Które kolumny są liczbowe?
2. Które kolumny są kategoryczne?
3. Która kolumna jest zmienną docelową?
4. Które kolumny mogą wymagać czyszczenia?

In [21]:
df.info()
df.describe(include='all')
'''
Ad. 1 'age', 'attendance_percent', 'weekly_study_hours',
       'assignment_score', 'quiz_score', 'previous_grade', 'forum_posts'
Ad. 2 żadne
Ad. 3 passed
Ad. 4 ?
'''

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   student_id          150 non-null    object 
 1   age                 150 non-null    int64  
 2   gender              150 non-null    object 
 3   city                149 non-null    object 
 4   program             150 non-null    object 
 5   study_mode          150 non-null    object 
 6   learning_platform   110 non-null    object 
 7   attendance_percent  149 non-null    float64
 8   weekly_study_hours  149 non-null    float64
 9   assignment_score    149 non-null    float64
 10  quiz_score          149 non-null    float64
 11  previous_grade      149 non-null    float64
 12  forum_posts         150 non-null    int64  
 13  project_submitted   149 non-null    object 
 14  passed              149 non-null    object 
dtypes: float64(5), int64(2), object(8)
memory usage: 17.7+ KB

,student_id,age,gender,city,program,study_mode,learning_platform,attendance_percent,weekly_study_hours,assignment_score,quiz_score,previous_grade,forum_posts,project_submitted,passed
count,150,150.000000,150,149,150,150,110,149.000000,149.000000,149.000000,149.00000,149.000000,150.00000,149,149
unique,149,NaN,4,8,7,4,4,NaN,NaN,NaN,NaN,NaN,NaN,3,3
top,STU0009,NaN,female,Wroclaw,Economics,full-time,Teams,NaN,NaN,NaN,NaN,NaN,NaN,yes,yes
freq,2,NaN,57,29,39,76,44,NaN,NaN,NaN,NaN,NaN,NaN,115,136
mean,NaN,24.746667,NaN,NaN,NaN,NaN,NaN,66.919463,9.119463,70.808054,62.07651,3.536913,8.84000,NaN,NaN
std,NaN,3.481855,NaN,NaN,NaN,NaN,NaN,19.022219,5.962061,19.900273,22.88801,0.890896,7.33666,NaN,NaN
min,NaN,19.000000,NaN,NaN,NaN,NaN,NaN,35.000000,-3.000000,29.400000,-10.00000,2.100000,-2.00000,NaN,NaN
25%,NaN,22.000000,NaN,NaN,NaN,NaN,NaN,52.000000,4.600000,58.500000,43.60000,2.700000,5.00000,NaN,NaN
50%,NaN,25.000000,NaN,NaN,NaN,NaN,NaN,65.000000,9.200000,72.000000,62.10000,3.500000,8.00000,NaN,NaN
75%,NaN,27.750000,NaN,NaN,NaN,NaN,NaN,81.000000,13.500000,83.700000,79.60000,4.200000,12.00000,NaN,NaN


## 4. Sprawdzenie zmiennej docelowej

Sprawdź, jakie wartości występują w kolumnie `passed`.

Zwróć uwagę także na braki danych i niespójne zapisy.

**Pytanie:** Czy klasy są silnie niezbalansowane? (tj. jedna klasa jest dużo liczniejsza od drugiej)

## 5. Sprawdzenie braków danych

Sprawdź:
- liczbę braków danych w każdej kolumnie,
- procent braków danych w każdej kolumnie,
- wiersze, w których występuje przynajmniej jeden brak.

**Pytanie:** Dlaczego braki danych są problemem przed trenowaniem modelu?

## 6. Sprawdzenie wartości w kolumnach tekstowych

Przed czyszczeniem sprawdź wartości w kolumnach tekstowych:
`gender`, `city`, `program`, `study_mode`, `learning_platform`, `project_submitted`, `passed`.

Użyj `value_counts(dropna=False)`.

**Pytanie:** Jakie niespójności widzisz w danych?

## 7. Czyszczenie kolumn tekstowych

Wyczyść kolumny tekstowe:
1. Zamień wartości na typ tekstowy.
2. Usuń zbędne spacje z początku i końca tekstu.
3. Zamień litery na małe.
4. Ujednolić zapisy kategorii, np. `full-time` i `full time` → `full_time`, `data analytics` → `data_analytics`, `google classroom` → `google_classroom`.

Po czyszczeniu ponownie sprawdź wartości w kolumnach tekstowych.

## 8. Konwersja kolumn liczbowych

Zdefiniuj listę kolumn liczbowych: `age`, `attendance_percent`, `weekly_study_hours`, `assignment_score`, `quiz_score`, `previous_grade`, `forum_posts`.

Następnie przekonwertuj je na wartości liczbowe za pomocą `pd.to_numeric(..., errors="coerce")`.

Sprawdź wynik za pomocą `info()`.

## 9. Znalezienie wartości błędnych i nierealnych

Znajdź wartości, które są nielogiczne lub niemożliwe, np.:
- `attendance_percent` poniżej 0 lub powyżej 100,
- `weekly_study_hours` poniżej 0,
- `assignment_score` poniżej 0 lub powyżej 100,
- `quiz_score` poniżej 0 lub powyżej 100,
- `previous_grade` poza zakresem 2–5,
- `forum_posts` poniżej 0.

Wyświetl wiersze zawierające takie wartości.

## 10. Poprawa wartości błędnych

Zamień wartości nierealne na `np.nan`.

Następnie sprawdź ponownie liczbę braków danych w każdej kolumnie.

**Pytanie:** Dlaczego lepiej zamienić wartość nierealną na brak danych niż zostawić ją w zbiorze?

## 11. Duplikaty

Sprawdź:
- liczbę duplikatów całych wierszy,
- duplikaty po kolumnie `student_id`.

Następnie usuń duplikaty po `student_id`, zachowując pierwszy rekord.

**Pytanie:** Dlaczego identyfikator studenta powinien być unikalny?

## 12. Braki w zmiennej docelowej

Sprawdź, czy w kolumnie `passed` występują braki danych.

Usuń rekordy bez wartości zmiennej docelowej.

**Pytanie:** Dlaczego w klasyfikacji zwykle usuwamy rekordy bez znanej klasy `y`?

## 13. Uzupełnianie braków w cechach

Uzupełnij braki danych:
- w kolumnach liczbowych medianą,
- w kolumnach kategorycznych wartością `unknown`.

Następnie sprawdź, czy w danych nadal występują braki.

## 14. Prosta eksploracja danych

Sprawdź:
1. Rozkład klas w kolumnie `passed`.
2. Średnie wartości według klasy dla kolumn: `attendance_percent`, `weekly_study_hours`, `assignment_score`, `quiz_score`.

Narysuj co najmniej jeden wykres pokazujący różnice między klasami.

## 15. Przygotowanie `X` i `y`

Przygotuj:
- `X` — cechy wejściowe modelu,
- `y` — zmienną docelową.

Usuń z `X` kolumny: `student_id`, `passed`.

Wyświetl kilka pierwszych wierszy `X` i `y`.

## 16. Podział na zbiór treningowy i testowy

Podziel dane na zbiór treningowy i testowy.

Użyj: `test_size=0.2`, `random_state=42`, `stratify=y`.

Następnie sprawdź rozkład klas w `y_train` i `y_test`.

**Pytanie:** Po co używamy `stratify=y`?

## 17. Preprocessing danych

Zdefiniuj osobno listę cech liczbowych i listę cech kategorycznych.

Następnie utwórz `ColumnTransformer`, który:
- dla cech liczbowych używa `StandardScaler`,
- dla cech kategorycznych używa `OneHotEncoder(handle_unknown="ignore")`.

**Pytanie:** Dlaczego modele `scikit-learn` nie mogą bezpośrednio pracować na tekście w kolumnach kategorycznych?

## 18. Budowa modelu jako `Pipeline`

Utwórz model jako `Pipeline`, który składa się z dwóch kroków:
1. `preprocessor`,
2. `LogisticRegression(max_iter=1000)`.

Następnie wyświetl obiekt modelu.

## 19. Trenowanie modelu

Wytrenuj model na danych treningowych. Użyj `fit()`.

**Pytanie:** Co oznacza trenowanie modelu?

## 20. Predykcja

Wykonaj predykcję dla danych testowych.

Następnie przygotuj tabelę zawierającą wartość rzeczywistą oraz wartość przewidzianą przez model.

Wyświetl pierwsze 10 wierszy takiej tabeli.

## 21. Accuracy

Policz `accuracy` dla modelu.

Wyświetl wynik w czytelnej formie, np. z trzema miejscami po przecinku.

**Pytanie:** Jak interpretujesz uzyskaną wartość accuracy?

## 22. Macierz pomyłek

Policz macierz pomyłek dla etykiet:

```python
labels=["no", "yes"]
```

Następnie narysuj prostą wizualizację macierzy pomyłek.

**Pytanie:** Jakiego rodzaju błędy popełnia model?

## 23. Classification report

Wyświetl `classification_report`.

Zwróć uwagę na: `precision`, `recall`, `f1-score`, `support`.

**Pytanie:** Czy model równie dobrze rozpoznaje obie klasy?

## 24. Predykcja prawdopodobieństw

Sprawdź prawdopodobieństwa klas dla danych testowych za pomocą `predict_proba()`.

Następnie:
1. Sprawdź kolejność klas przez `model.classes_`.
2. Utwórz tabelę z prawdopodobieństwami dla klas `no` i `yes`.
3. Połącz ją z tabelą wyników rzeczywistych i przewidywanych.

**Pytanie:** Czym różni się predykcja klasy od predykcji prawdopodobieństwa?

## 25. Predykcja dla nowego studenta

Utwórz własny przykład nowego studenta jako `DataFrame`.

Dane muszą zawierać te same kolumny co `X`, bez `student_id` i bez `passed`.

Następnie sprawdź:
- przewidywaną klasę,
- prawdopodobieństwa klas.

**Pytanie:** Czy taki model powinien być używany jako automatyczna decyzja o zaliczeniu? Uzasadnij odpowiedź.